# Online Retail — Câu 9: Tìm tập phổ biến bằng Apriori

**Võ Thiện Linh — MSSV: 6451071041**

Áp dụng Apriori trên các giỏ hàng `United Kingdom`, đã loại hóa đơn tín dụng, chỉ giữ mặt hàng có tổng `Quantity > 0` và bỏ `POSTAGE`. Ngưỡng hỗ trợ tối thiểu là 3%. Cài đặt Apriori trực tiếp bằng Python để không cần thư viện bổ sung.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
from collections import Counter
from itertools import combinations
import math
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
uk = df.loc[df['Country'].eq('United Kingdom')]
grouped = uk.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_ids = pd.Index(grouped['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
purchases = grouped.loc[grouped['Quantity'].gt(0) & grouped['Description'].ne('POSTAGE')]
items_by_invoice = purchases.groupby('InvoiceNo')['Description'].agg(frozenset).to_dict()
transactions = [items_by_invoice.get(invoice, frozenset()) for invoice in invoice_ids]
def apriori_counts(transactions, min_support):
    minimum_count = math.ceil(min_support * len(transactions))
    item_counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in item_counts.items() if count >= minimum_count}
    frequent_counts = dict(previous)
    frequent_items = set(item_counts)
    size = 2
    while previous:
        previous_list = sorted(previous, key=lambda itemset: tuple(sorted(itemset)))
        previous_set = set(previous)
        candidates = set()
        for i, left in enumerate(previous_list):
            for right in previous_list[i + 1:]:
                candidate = left | right
                if len(candidate) == size and all(frozenset(part) in previous_set for part in combinations(candidate, size - 1)):
                    candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            active_items = sorted(transaction & frequent_items)
            for combination in combinations(active_items, size):
                itemset = frozenset(combination)
                if itemset in candidates:
                    candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= minimum_count}
        frequent_counts.update(previous)
        size += 1
    return frequent_counts, minimum_count
min_support = 0.03
frequent_counts, minimum_count = apriori_counts(transactions, min_support)
frequent_itemsets = pd.DataFrame([{'itemsets': ', '.join(sorted(itemset)), 'itemset_size': len(itemset), 'support': count / len(transactions), 'invoice_count': count} for itemset, count in frequent_counts.items()])
frequent_itemsets = frequent_itemsets.sort_values(['itemset_size', 'support'], ascending=[True, False]).reset_index(drop=True)
print(f'Số hóa đơn trong mẫu: {len(transactions):,}')
print(f'Minimum support: {min_support:.0%} ({minimum_count:,} hóa đơn)')
print(f'Số tập phổ biến: {len(frequent_itemsets):,}')
display(frequent_itemsets)